# Week 5 Breakout Slides: Decision Tree Regression

Linear regression assumes the target moves in a straight line with the features. A **decision tree** makes no such assumption: it repeatedly splits the data on the feature and threshold that best reduce error, carving the feature space into regions and predicting a constant value within each. This lets it capture nonlinear relationships and interactions automatically. We tackle the same task — predicting life expectancy — so the two approaches can be compared directly.

## Import Libraries

We start with the familiar trio — `numpy`, `pandas`, and `matplotlib`. The `scikit-learn` pieces (the model, the train/test split, and the evaluation metrics) are imported later, each right before we use it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Load and Inspect the Data

We reuse the same country development dataset from Week 4 — one row per country, one column per indicator, all numeric.

In [ ]:
path_to_file = Path('your/path')

In [ ]:
df = pd.read_csv(path_to_file / 'country_data.csv')

### Inspect the DataFrame

A quick look confirms the columns and shape before we start modeling.

In [ ]:
df.columns

## Define Features and Target

Unlike the unsupervised work in Week 4, regression is **supervised**: we single out one column to predict (the **target**, `y`) and use the rest as inputs (the **features**, `X`). Here we predict **life expectancy** from a country's other development indicators.

In [ ]:
target = 'life_expec'

X = df.drop(columns=target)
y = df[target]

## Train/Test Split

To judge how well the model generalizes, we hold out part of the data. We fit on the **training set** and evaluate on the **test set** the model never saw — this is what catches overfitting. `random_state` makes the split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Fit a Decision Tree Regressor

`DecisionTreeRegressor` grows the tree by greedily choosing splits. Left unchecked it will grow until it memorizes the training data, so we cap `max_depth` to keep the tree shallow, interpretable, and less prone to overfitting. We start with a depth of 3.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

## Evaluate the Model

With a fitted model in hand, we predict on the held-out test set and quantify how close those predictions are to the truth.

### Predictions

`.predict()` returns the model's estimated life expectancy for each country in the test set.

In [ ]:
y_pred = model.predict(X_test)

### Metrics

Three standard regression scores summarize performance:

- **R²** — the fraction of variance in the target the model explains (1.0 is perfect, 0.0 is no better than predicting the mean).
- **MAE** — mean absolute error, the average miss in the target's own units (years).
- **RMSE** — root mean squared error, like MAE but penalizes large misses more heavily.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)

print(f"R^2:  {r2:.3f}")
print(f"MAE:  {mae:.3f} years")

### Predicted vs. Actual

Plotting predictions against the true values is the quickest visual gut-check. Points hugging the dashed 45-degree line mean the model is on target; systematic departures reveal where it struggles.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, color='steelblue', alpha=0.8)

# 45-degree reference line: perfect predictions land here
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', label='Perfect prediction')

ax.set_xlabel('Actual Life Expectancy (years)')
ax.set_ylabel('Predicted Life Expectancy (years)')
ax.set_title('Predicted vs. Actual')
ax.legend()
ax.grid(alpha=0.3)

### Residual Plot

A **residual** is the leftover error (actual minus predicted). Plotting residuals against the predictions should look like a shapeless cloud centered on zero. Curves or a fanning-out pattern signal that the model is missing structure in the data.

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_pred, residuals, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', linestyle='--')

ax.set_xlabel('Predicted Life Expectancy (years)')
ax.set_ylabel('Residual (actual - predicted)')
ax.set_title('Residual Plot')
ax.grid(alpha=0.3)

## Interpreting the Model

A tree is interpretable in two complementary ways: which features it relied on most, and the exact decision path it follows.

### Feature Importances

`feature_importances_` reports how much each feature reduced prediction error across all the splits that used it, normalized to sum to 1. Higher means more influential.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot.barh(color='steelblue', edgecolor='k', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('Decision Tree Feature Importances')
ax.grid(alpha=0.3)

### Visualizing the Tree

Because the tree is shallow, we can draw it in full. Each box shows the split rule, the number of samples that reach it, and the predicted value — read top to bottom to trace how a prediction is made.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(model,
          feature_names=X.columns,
          filled=True,
          rounded=True,
          fontsize=9,
          ax=ax)
ax.set_title('Decision Tree (max_depth=3)')
plt.show()

## Diagnostic: Tuning Tree Depth

How deep should the tree be? Too shallow and it underfits; too deep and it overfits, memorizing the training set at the expense of new data. We sweep `max_depth` and compare training vs. test R². The gap between the two curves is the tell-tale sign of overfitting: training accuracy keeps climbing while test accuracy stalls or drops.

In [ ]:
depths = range(1, 11)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(depths), train_scores, marker='o', label='Train R^2', color='steelblue')
ax.plot(list(depths), test_scores, marker='o', label='Test R^2', color='darkorange')

ax.set_xticks(list(depths))
ax.set_xlabel('max_depth')
ax.set_ylabel('R^2')
ax.set_title('Training vs. Test Performance by Tree Depth')
ax.legend()
ax.grid(alpha=0.3)